<a href="https://colab.research.google.com/github/amirhamrouni/Alala/blob/main/docs/MoneyPrinterTurbo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoneyPrinterTurbo Automation Stack\nخلية واحدة تشغل MoneyPrinterTurbo API + WebUI + إصلاح العربية RTL + Queue Gateway + Telegram long-poll. لا يلزم VPS أو SSH. ضع الأسرار في Colab Secrets: `NGROK_AUTHTOKEN`, `TELEGRAM_BOT_TOKEN`, `PEXELS_API_KEY`، و`OPENROUTER_API_KEY` اختياري.

In [ ]:
import os, re, subprocess, time, urllib.request, textwrap, json\nfrom pathlib import Path\nfrom getpass import getpass\n\nMPT = Path('/content/MoneyPrinterTurbo')\nALALA = Path('/content/Alala')\nMPT_UPSTREAM = 'https://github.com/harry0703/MoneyPrinterTurbo.git'\nALALA_UPSTREAM = 'https://github.com/amirhamrouni/Alala.git'\nPATCH_URL = 'https://raw.githubusercontent.com/amirhamrouni/Alala/main/moneyprinter-mobile-gateway/scripts/patch_moneyprinter_arabic.py'\n\nAPI_PORT = 8090\nWEB_PORT = 8501\nGATEWAY_PORT = 8000\n\ndef run(cmd, **kwargs):\n    print('$', ' '.join(map(str, cmd)), flush=True)\n    return subprocess.run(cmd, check=True, **kwargs)\n\ndef healthy(url):\n    try:\n        with urllib.request.urlopen(url, timeout=3) as r:\n            return r.status == 200\n    except Exception:\n        return False\n\ndef secret(name):\n    try:\n        from google.colab import userdata\n        value = userdata.get(name)\n        return value.strip() if value else ''\n    except Exception:\n        return ''\n\nrun(['python', '-m', 'pip', 'install', '-q', 'uv', 'pyngrok', 'requests'])\n\nfor repo, upstream in ((MPT, MPT_UPSTREAM), (ALALA, ALALA_UPSTREAM)):\n    if (repo / '.git').is_dir():\n        run(['git', '-C', str(repo), 'pull', '--ff-only'])\n    else:\n        if repo.exists():\n            run(['rm', '-rf', str(repo)])\n        run(['git', 'clone', '--depth', '1', upstream, str(repo)])\n\nrun(['uv', 'python', 'install', '3.11'])\nrun(['uv', 'sync', '--frozen', '--python', '3.11'], cwd=MPT)\nrun(['python', '-m', 'pip', 'install', '-q', '-r', str(ALALA / 'moneyprinter-mobile-gateway/requirements.txt')])\n\npatch_file = Path('/content/patch_moneyprinter_arabic.py')\nurllib.request.urlretrieve(PATCH_URL, patch_file)\nrun(['python', str(patch_file), '--repo', str(MPT)])\n\ncfg = MPT / 'config.toml'\nif not cfg.exists():\n    cfg.write_text((MPT / 'config.example.toml').read_text(encoding='utf-8'), encoding='utf-8')\nraw = cfg.read_text(encoding='utf-8')\n\ndef set_toml_scalar(text, key, value_literal):\n    pattern = rf'(?m)^{re.escape(key)}\s*=\s*.*$'\n    replacement = f'{key} = {value_literal}'\n    if re.search(pattern, text):\n        return re.sub(pattern, replacement, text, count=1)\n    return text + '\n' + replacement + '\n'\n\nraw = set_toml_scalar(raw, 'listen_port', str(API_PORT))\nraw = set_toml_scalar(raw, 'video_source', '"pexels"')\n\npexels_key = secret('PEXELS_API_KEY') or secret('MPT_PEXELS_API_KEY')\nif pexels_key:\n    raw = set_toml_scalar(raw, 'pexels_api_keys', json.dumps([pexels_key]))\n\nopenrouter_key = secret('OPENROUTER_API_KEY')\nif openrouter_key:\n    raw = set_toml_scalar(raw, 'llm_provider', '"openrouter"')\n    raw = set_toml_scalar(raw, 'openrouter_api_key', json.dumps(openrouter_key))\n    openrouter_model = secret('OPENROUTER_MODEL') or 'minimax/minimax-m3:free'\n    raw = set_toml_scalar(raw, 'openrouter_model_name', json.dumps(openrouter_model))\n\ncfg.write_text(raw, encoding='utf-8')\n\nfor port in (API_PORT, WEB_PORT, GATEWAY_PORT):\n    subprocess.run(['bash', '-lc', f'fuser -k {port}/tcp >/dev/null 2>&1 || true'], check=False)\ntime.sleep(1)\n\napi_log = open('/content/mpt-api.log', 'w', encoding='utf-8')\nweb_log = open('/content/mpt-webui.log', 'w', encoding='utf-8')\n\napi_proc = subprocess.Popen(['uv', 'run', 'python', 'main.py'], cwd=MPT, stdout=api_log, stderr=subprocess.STDOUT)\nweb_proc = subprocess.Popen([\n    'uv','run','streamlit','run','webui/Main.py',\n    f'--server.port={WEB_PORT}','--server.address=0.0.0.0',\n    '--browser.gatherUsageStats=False','--client.toolbarMode=minimal',\n    '--server.showEmailPrompt=False'\n], cwd=MPT, stdout=web_log, stderr=subprocess.STDOUT)\n\nfor _ in range(60):\n    if healthy(f'http://127.0.0.1:{API_PORT}/ping') and healthy(f'http://127.0.0.1:{WEB_PORT}/_stcore/health'):\n        break\n    time.sleep(2)\nelse:\n    api_log.flush(); web_log.flush()\n    print(Path('/content/mpt-api.log').read_text(errors='replace')[-3000:])\n    print(Path('/content/mpt-webui.log').read_text(errors='replace')[-3000:])\n    raise RuntimeError('MoneyPrinterTurbo did not become healthy')\n\ntg_token = secret('TELEGRAM_BOT_TOKEN')\ngateway_env = os.environ.copy()\ngateway_env.update({\n    'MONEYPRINTER_BASE_URL': f'http://127.0.0.1:{API_PORT}',\n    'MONEYPRINTER_STORAGE_ROOT': str(MPT),\n    'JOB_STATE_PATH': '/content/mpt-gateway-data/jobs.json',\n    'DEFAULT_LANGUAGE': 'ar',\n    'DEFAULT_ASPECT': '9:16',\n    'DEFAULT_VIDEO_SOURCE': 'pexels',\n    'DEFAULT_VIDEO_CONCAT_MODE': 'random',\n    'DEFAULT_VIDEO_CLIP_DURATION': '3',\n    'DEFAULT_VOICE_NAME': 'ar-SA-ZariyahNeural',\n    'DEFAULT_FONT_NAME': 'DejaVuSans.ttf',\n    'TELEGRAM_SEND_VIDEO': 'true',\n})\nif tg_token:\n    gateway_env['TELEGRAM_BOT_TOKEN'] = tg_token\n\ngateway_log = open('/content/mpt-gateway.log', 'w', encoding='utf-8')\ngateway_proc = subprocess.Popen(\n    ['python', '-m', 'uvicorn', 'app.main:app',\n     '--app-dir', str(ALALA / 'moneyprinter-mobile-gateway'),\n     '--host', '127.0.0.1', '--port', str(GATEWAY_PORT)],\n    cwd=ALALA / 'moneyprinter-mobile-gateway',\n    env=gateway_env,\n    stdout=gateway_log, stderr=subprocess.STDOUT\n)\n\nfor _ in range(30):\n    if healthy(f'http://127.0.0.1:{GATEWAY_PORT}/health'):\n        break\n    time.sleep(1)\nelse:\n    gateway_log.flush()\n    print(Path('/content/mpt-gateway.log').read_text(errors='replace')[-3000:])\n    raise RuntimeError('Gateway did not become healthy')\n\nif tg_token:\n    poller_path = Path('/content/telegram_poller.py')\n    poller_path.write_text(textwrap.dedent(f"""\n        import os, time, requests\n\n        token = os.environ['TELEGRAM_BOT_TOKEN']\n        tg = f'https://api.telegram.org/bot{{token}}'\n        gateway = 'http://127.0.0.1:{GATEWAY_PORT}/telegram/webhook'\n        s = requests.Session()\n\n        s.post(tg + '/deleteWebhook', json={{'drop_pending_updates': False}}, timeout=30)\n        me = s.get(tg + '/getMe', timeout=30).json()\n        if not me.get('ok'):\n            raise RuntimeError('Telegram token rejected')\n        print('Telegram ready: @' + me['result']['username'], flush=True)\n\n        offset = 0\n        while True:\n            try:\n                data = s.get(tg + '/getUpdates', params={{'offset': offset, 'timeout': 25}}, timeout=35).json()\n                for update in data.get('result', []):\n                    offset = update['update_id'] + 1\n                    r = s.post(gateway, json=update, timeout=30)\n                    r.raise_for_status()\n            except Exception as exc:\n                print('poll error:', type(exc).__name__, str(exc)[:200], flush=True)\n                time.sleep(5)\n    """), encoding='utf-8')\n    poller_log = open('/content/telegram-poller.log', 'w', encoding='utf-8')\n    telegram_poller_proc = subprocess.Popen(\n        ['python', str(poller_path)],\n        env={**os.environ, 'TELEGRAM_BOT_TOKEN': tg_token},\n        stdout=poller_log, stderr=subprocess.STDOUT\n    )\n    time.sleep(2)\nelse:\n    telegram_poller_proc = None\n    print('TELEGRAM_BOT_TOKEN not found in Colab Secrets; Telegram bridge skipped.')\n\nngrok_token = secret('NGROK_AUTHTOKEN')\nif not ngrok_token:\n    ngrok_token = getpass('NGROK_AUTHTOKEN: ').strip()\n\nif ngrok_token:\n    from pyngrok import ngrok\n    ngrok.set_auth_token(ngrok_token)\n    try:\n        ngrok.kill()\n    except Exception:\n        pass\n    web_tunnel = ngrok.connect(addr=f'http://127.0.0.1:{WEB_PORT}', proto='http', bind_tls=True)\n    print('WEBUI =', web_tunnel.public_url)\nelse:\n    print('NGROK_AUTHTOKEN missing; WebUI remains local.')\n\nprint('API_LOCAL =', f'http://127.0.0.1:{API_PORT}')\nprint('GATEWAY_LOCAL =', f'http://127.0.0.1:{GATEWAY_PORT}')\nprint('Arabic subtitles = RTL patched / DejaVuSans')\nprint('Queue + dedupe + job_id = enabled')\nprint('Telegram MP4 upload = enabled when TELEGRAM_BOT_TOKEN exists')\nprint('READY')\n

## اختبار الحالة\nبعد ظهور `READY`، أرسل للبوت: `اعمل فيديو عن فوائد الذكاء الاصطناعي في الدراسة`. البوت يضع الطلب في Queue ثم يعيد ملف MP4 لنفس المحادثة.

In [ ]:
import requests, json, pathlib\n\nchecks = {\n    'mpt_api': requests.get('http://127.0.0.1:8090/ping', timeout=10).status_code,\n    'webui': requests.get('http://127.0.0.1:8501/_stcore/health', timeout=10).status_code,\n    'gateway': requests.get('http://127.0.0.1:8000/health', timeout=10).json(),\n}\nprint(json.dumps(checks, ensure_ascii=False, indent=2))\n\nfor path in ('/content/mpt-api.log', '/content/mpt-webui.log', '/content/mpt-gateway.log', '/content/telegram-poller.log'):\n    p = pathlib.Path(path)\n    if p.exists():\n        print('\n---', p.name, '---')\n        print(p.read_text(errors='replace')[-1200:])\n